# Software FJ – Fase 4
Programación 213023 · UNAD
Cuaderno fuente. Ejecutar las celdas en orden para crear los módulos, ejecutar las simulaciones y comprobar las pruebas. Se puede abrir el menú de forma opcional.

## modelos.py

In [ ]:
%%writefile modelos.py
"""Clases del ejercicio Software FJ (Programación, fase 4)."""
from abc import ABC, abstractmethod
import math
import re


class ErrorFJ(Exception):
    """Error general de la aplicación."""


class DatoInvalido(ErrorFJ):
    pass


class NoDisponible(ErrorFJ):
    pass


class EstadoIncorrecto(ErrorFJ):
    pass


class NoEncontrado(ErrorFJ):
    pass


class Duplicado(ErrorFJ):
    pass


def texto(valor, campo):
    # Evita valores vacíos y datos de tipo incorrecto.
    if not isinstance(valor, str) or not valor.strip():
        raise DatoInvalido(f"{campo} es obligatorio")
    return valor.strip()


def numero(valor, campo, minimo=0, maximo=None):
    try:
        if isinstance(valor, bool):
            raise TypeError("No se admite un booleano")
        valor = float(valor)
    except (ValueError, TypeError, OverflowError) as error:
        raise DatoInvalido(f"{campo} debe ser numérico") from error
    if not math.isfinite(valor) or valor < minimo:
        raise DatoInvalido(f"{campo} fuera del rango permitido")
    if maximo is not None and valor > maximo:
        raise DatoInvalido(f"{campo} supera el máximo")
    return valor


def positivo(valor, campo):
    valor = numero(valor, campo)
    if valor <= 0:
        raise DatoInvalido(f"{campo} debe ser mayor que cero")
    return valor


class Entidad(ABC):
    def __init__(self, codigo):
        self.__codigo = texto(codigo, "Código")

    @property
    def codigo(self):
        return self.__codigo

    @abstractmethod
    def describir(self):
        pass


class Cliente(Entidad):
    def __init__(self, documento, nombre, correo, telefono):
        super().__init__(documento)
        if not re.fullmatch(r"[A-Za-z0-9]{5,20}", self.codigo):
            raise DatoInvalido("Documento inválido")
        self.nombre = nombre
        self.correo = correo
        self.telefono = telefono

    @property
    def nombre(self):
        return self.__nombre

    @nombre.setter
    def nombre(self, valor):
        valor = texto(valor, "Nombre")
        if len(valor) < 3 or not any(c.isalpha() for c in valor):
            raise DatoInvalido("Nombre inválido")
        self.__nombre = valor

    @property
    def correo(self):
        return self.__correo

    @correo.setter
    def correo(self, valor):
        valor = texto(valor, "Correo")
        if not re.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", valor):
            raise DatoInvalido("Correo inválido")
        self.__correo = valor

    @property
    def telefono(self):
        return self.__telefono

    @telefono.setter
    def telefono(self, valor):
        valor = texto(valor, "Teléfono")
        if not re.fullmatch(r"\+?\d{7,15}", valor):
            raise DatoInvalido("Teléfono inválido")
        self.__telefono = valor

    def describir(self):
        return f"{self.codigo} - {self.nombre}"


class Servicio(Entidad, ABC):
    def __init__(self, codigo, nombre, tarifa):
        super().__init__(codigo)
        self.__nombre = texto(nombre, "Servicio")
        self.__tarifa = positivo(tarifa, "Tarifa")
        self.__disponible = True

    @property
    def nombre(self):
        return self.__nombre

    @property
    def tarifa(self):
        return self.__tarifa

    @property
    def disponible(self):
        return self.__disponible

    def ocupar(self):
        if not self.disponible:
            raise NoDisponible(f"{self.codigo} no está disponible")
        self.__disponible = False

    def liberar(self):
        self.__disponible = True

    @abstractmethod
    def validar_parametros(self, duracion):
        pass

    @abstractmethod
    def costo_base(self, duracion):
        pass

    # En Python se simulan variantes de un método con parámetros opcionales.
    def calcular_costo(self, duracion, impuesto=0, descuento=0):
        duracion = self.validar_parametros(duracion)
        impuesto = numero(impuesto, "Impuesto", 0, 100)
        descuento = numero(descuento, "Descuento", 0, 100)
        subtotal = self.costo_base(duracion)
        total = subtotal * (1 - descuento / 100) * (1 + impuesto / 100)
        if not math.isfinite(total):
            raise DatoInvalido("El cálculo del costo no es válido")
        return round(total, 2)


class Sala(Servicio):
    def __init__(self, codigo, nombre, tarifa, capacidad):
        super().__init__(codigo, nombre, tarifa)
        capacidad = positivo(capacidad, "Capacidad")
        if not capacidad.is_integer():
            raise DatoInvalido("La capacidad debe ser entera")
        self.capacidad = int(capacidad)

    def validar_parametros(self, duracion):
        return positivo(duracion, "Horas de sala")

    def costo_base(self, duracion):
        return self.tarifa * duracion

    def describir(self):
        return f"{self.codigo} - Sala {self.nombre} ({self.capacidad} personas)"


class Equipo(Servicio):
    def __init__(self, codigo, nombre, tarifa, tipo):
        super().__init__(codigo, nombre, tarifa)
        self.tipo = texto(tipo, "Tipo de equipo")

    def validar_parametros(self, duracion):
        dias = positivo(duracion, "Días de alquiler")
        if not dias.is_integer():
            raise DatoInvalido("El alquiler se calcula por días completos")
        return dias

    def costo_base(self, duracion):
        return self.tarifa * duracion

    def describir(self):
        return f"{self.codigo} - Equipo {self.nombre} ({self.tipo})"


class Asesoria(Servicio):
    def __init__(self, codigo, nombre, tarifa, especialidad):
        super().__init__(codigo, nombre, tarifa)
        self.especialidad = texto(especialidad, "Especialidad")

    def validar_parametros(self, duracion):
        horas = positivo(duracion, "Horas de asesoría")
        if horas < 1:
            raise DatoInvalido("La asesoría requiere al menos una hora")
        return horas

    def costo_base(self, duracion):
        return self.tarifa * duracion

    def describir(self):
        return f"{self.codigo} - Asesoría {self.nombre} ({self.especialidad})"


class Reserva(Entidad):
    def __init__(self, codigo, cliente, servicio, duracion):
        super().__init__(codigo)
        if not isinstance(cliente, Cliente) or not isinstance(servicio, Servicio):
            raise DatoInvalido("La reserva necesita un cliente y un servicio")
        self.cliente = cliente
        self.servicio = servicio
        self.duracion = servicio.validar_parametros(duracion)
        self.__estado = "Pendiente"
        self.__total = None

    @property
    def estado(self):
        return self.__estado

    @property
    def total(self):
        return self.__total

    def confirmar(self):
        if self.estado != "Pendiente":
            raise EstadoIncorrecto("Solo se confirman reservas pendientes")
        self.servicio.ocupar()
        self.__estado = "Confirmada"

    def cancelar(self):
        if self.estado not in ("Pendiente", "Confirmada"):
            raise EstadoIncorrecto("La reserva no puede cancelarse")
        if self.estado == "Confirmada":
            self.servicio.liberar()
        self.__estado = "Cancelada"

    def procesar(self, impuesto=0, descuento=0):
        if self.estado != "Confirmada":
            raise EstadoIncorrecto("Primero se debe confirmar la reserva")
        # Si falla el cálculo, la reserva sigue confirmada.
        try:
            total = self.servicio.calcular_costo(self.duracion, impuesto, descuento)
        except DatoInvalido as error:
            raise DatoInvalido("No se pudo calcular el total de la reserva") from error
        else:
            self.__total = total
            self.__estado = "Procesada"
            self.servicio.liberar()
            return total

    def describir(self):
        return f"{self.codigo} - {self.cliente.nombre} / {self.servicio.nombre} / {self.estado}"


## gestor.py

In [ ]:
%%writefile gestor.py
"""Administración de las listas y registro de operaciones."""
import logging
from pathlib import Path
from modelos import Cliente, Servicio, Reserva, ErrorFJ, Duplicado, NoEncontrado, NoDisponible, DatoInvalido


def configurar_log(ruta="logs/software_fj.log"):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("software_fj." + str(ruta.resolve()))
    logger.setLevel(logging.INFO)
    logger.propagate = False
    if not logger.handlers:
        archivo = logging.FileHandler(ruta, encoding="utf-8")
        archivo.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
        logger.addHandler(archivo)
    return logger


class Gestor:
    def __init__(self):
        # Las listas existen solo durante la ejecución.
        self.clientes = []
        self.servicios = []
        self.reservas = []

    def buscar(self, lista, codigo):
        for elemento in lista:
            if elemento.codigo == codigo:
                return elemento
        raise NoEncontrado(f"No existe el código {codigo}")

    def agregar(self, lista, elemento):
        if any(item.codigo == elemento.codigo for item in lista):
            raise Duplicado(f"El código {elemento.codigo} ya existe")
        lista.append(elemento)
        return elemento

    def registrar_cliente(self, documento, nombre, correo, telefono):
        return self.agregar(self.clientes, Cliente(documento, nombre, correo, telefono))

    def registrar_servicio(self, servicio):
        if not isinstance(servicio, Servicio):
            raise DatoInvalido("Se requiere un objeto de tipo Servicio")
        return self.agregar(self.servicios, servicio)

    def crear_reserva(self, codigo, documento, servicio, duracion):
        cliente = self.buscar(self.clientes, documento)
        servicio = self.buscar(self.servicios, servicio)
        if not servicio.disponible:
            raise NoDisponible("El servicio ya está ocupado")
        return self.agregar(self.reservas, Reserva(codigo, cliente, servicio, duracion))

    def confirmar(self, codigo):
        self.buscar(self.reservas, codigo).confirmar()

    def cancelar(self, codigo):
        self.buscar(self.reservas, codigo).cancelar()

    def procesar(self, codigo, impuesto=0, descuento=0):
        return self.buscar(self.reservas, codigo).procesar(impuesto, descuento)


def ejecutar(logger, descripcion, operacion):
    """Captura y registra el error sin detener las siguientes operaciones."""
    try:
        resultado = operacion()
    except ErrorFJ as error:
        logger.exception("%s: %s", descripcion, error)
        return False, error
    except Exception as error:
        # Última barrera para errores inesperados de una operación.
        logger.exception("Error inesperado en %s", descripcion)
        return False, error
    else:
        logger.info("Correcto: %s", descripcion)
        return True, resultado
    finally:
        logger.info("Fin de operación: %s", descripcion)


## simulaciones.py

In [ ]:
%%writefile simulaciones.py
"""Casos de prueba con datos ficticios."""
from modelos import Sala, Equipo, Asesoria
from gestor import Gestor, configurar_log, ejecutar


def simular(logger=None):
    logger = logger or configurar_log()
    g = Gestor()
    # Cada caso indica si debe funcionar (True) o rechazarse (False).
    casos = [
        ("Cliente válido", True, lambda: g.registrar_cliente("10001", "Ana Pérez", "ana@example.com", "3001234567")),
        ("Cliente con correo incorrecto", False, lambda: g.registrar_cliente("10002", "Luis Díaz", "correo", "3001112222")),
        ("Cliente duplicado", False, lambda: g.registrar_cliente("10001", "Ana Pérez", "ana@example.com", "3001234567")),
        ("Segundo cliente válido", True, lambda: g.registrar_cliente("10003", "Luis Díaz", "luis@example.com", "3001112222")),
        ("Registrar sala", True, lambda: g.registrar_servicio(Sala("S1", "Sala principal", 50000, 12))),
        ("Tarifa negativa", False, lambda: g.registrar_servicio(Sala("S2", "Sala auxiliar", -10, 8))),
        ("Registrar equipo", True, lambda: g.registrar_servicio(Equipo("E1", "Portátil", 80000, "Computación"))),
        ("Registrar asesoría", True, lambda: g.registrar_servicio(Asesoria("A1", "Consultoría", 100000, "Software"))),
        ("Servicio con parámetro faltante", False, lambda: g.registrar_servicio(Asesoria("A2", "Consultoría", 100000, ""))),
        ("Reserva de sala", True, lambda: g.crear_reserva("R1", "10001", "S1", 2)),
        ("Confirmar reserva", True, lambda: g.confirmar("R1")),
        ("Servicio ocupado", False, lambda: g.crear_reserva("R2", "10003", "S1", 1)),
        ("Confirmar dos veces", False, lambda: g.confirmar("R1")),
        ("Descuento inválido", False, lambda: g.procesar("R1", 19, 120)),
        ("Procesar reserva con impuesto y descuento", True, lambda: g.procesar("R1", 19, 10)),
        ("Reserva de equipo", True, lambda: g.crear_reserva("R3", "10003", "E1", 3)),
        ("Procesar sin confirmar", False, lambda: g.procesar("R3")),
        ("Confirmar equipo", True, lambda: g.confirmar("R3")),
        ("Cancelar equipo", True, lambda: g.cancelar("R3")),
        ("Volver a reservar equipo", True, lambda: g.crear_reserva("R4", "10001", "E1", 1)),
        ("Asesoría con duración incorrecta", False, lambda: g.crear_reserva("R5", "10001", "A1", 0)),
        ("Cliente inexistente", False, lambda: g.crear_reserva("R6", "99999", "A1", 1)),
        ("Asesoría válida", True, lambda: g.crear_reserva("R7", "10001", "A1", 2)),
        ("Confirmar asesoría", True, lambda: g.confirmar("R7")),
        ("Procesar asesoría", True, lambda: g.procesar("R7")),
    ]
    correctos = 0
    print("\n--- Simulación Software FJ ---")
    for indice, (nombre, esperado, accion) in enumerate(casos, 1):
        exito, resultado = ejecutar(logger, nombre, accion)
        coincide = exito == esperado
        correctos += int(coincide)
        detalle = "Operación aceptada" if exito else str(resultado)
        print(f"{indice:02d}. {nombre}: {'OK' if coincide else 'REVISAR'} - {detalle}")
    print(f"Resultado: {correctos}/{len(casos)} casos coinciden con lo esperado.")
    return correctos, len(casos)


if __name__ == "__main__":
    correctos, total = simular()
    raise SystemExit(0 if correctos == total else 1)


## main.py

In [ ]:
%%writefile main.py
"""Menú de consola del ejercicio Software FJ."""
from modelos import Sala, Equipo, Asesoria
from gestor import Gestor, configurar_log, ejecutar
from simulaciones import simular


def alta_cliente(g):
    return g.registrar_cliente(input("Documento: "), input("Nombre: "),
                               input("Correo: "), input("Teléfono: "))


def alta_servicio(g):
    tipo = input("Tipo (1 sala, 2 equipo, 3 asesoría): ")
    clases = {"1": (Sala, "Capacidad"), "2": (Equipo, "Tipo de equipo"),
              "3": (Asesoria, "Especialidad")}
    if tipo not in clases:
        from modelos import DatoInvalido
        raise DatoInvalido("Tipo de servicio incorrecto")
    clase, campo = clases[tipo]
    codigo = input("Código: ")
    nombre = input("Nombre: ")
    tarifa = input("Tarifa por hora o día: ")
    return g.registrar_servicio(clase(codigo, nombre, tarifa, input(campo + ": ")))


def alta_reserva(g):
    return g.crear_reserva(input("Código de reserva: "), input("Documento del cliente: "),
                          input("Código de servicio: "), input("Duración (horas o días): "))


def listar(elementos):
    if not elementos:
        print("No hay registros.")
    for elemento in elementos:
        print(elemento.describir())


def main():
    logger = configurar_log()
    g = Gestor()
    opciones = {
        "1": ("Registrar cliente", lambda: alta_cliente(g)),
        "2": ("Listar clientes", lambda: listar(g.clientes)),
        "3": ("Registrar servicio", lambda: alta_servicio(g)),
        "4": ("Listar servicios", lambda: listar(g.servicios)),
        "5": ("Crear reserva", lambda: alta_reserva(g)),
        "6": ("Confirmar reserva", lambda: g.confirmar(input("Código: "))),
        "7": ("Cancelar reserva", lambda: g.cancelar(input("Código: "))),
        "8": ("Procesar reserva", lambda: g.procesar(input("Código: "), input("Impuesto (%): "), input("Descuento (%): "))),
        "9": ("Listar reservas", lambda: listar(g.reservas)),
        "10": ("Ejecutar simulaciones", lambda: simular(logger)),
    }
    while True:
        print("\n=== SOFTWARE FJ ===")
        for clave, (nombre, _) in opciones.items():
            print(f"{clave}. {nombre}")
        print("0. Salir")
        try:
            opcion = input("Seleccione: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nFin de la sesión.")
            break
        if opcion == "0":
            break
        if opcion not in opciones:
            logger.warning("Opción inválida del menú")
            print("Opción no válida.")
            continue
        nombre, accion = opciones[opcion]
        exito, resultado = ejecutar(logger, nombre, accion)
        if not exito:
            print("Error:", resultado)
        elif hasattr(resultado, "describir"):
            print(resultado.describir())
        elif isinstance(resultado, float):
            print(f"Total: {resultado:,.2f} COP")
        else:
            print("Operación terminada.")


if __name__ == "__main__":
    main()


## test_software_fj.py

In [ ]:
%%writefile test_software_fj.py
"""Pruebas unitarias. Ejecutar: python -m unittest -v"""
import tempfile
import unittest
from pathlib import Path
from modelos import (Entidad, Servicio, Cliente, Sala, Equipo, Asesoria,
                     DatoInvalido, Duplicado, NoDisponible, NoEncontrado, EstadoIncorrecto)
from gestor import Gestor, configurar_log, ejecutar
from simulaciones import simular


class PruebasSoftwareFJ(unittest.TestCase):
    def setUp(self):
        self.g = Gestor()
        self.cliente = self.g.registrar_cliente("12345", "Ana Pérez", "ana@example.com", "3001234567")
        self.sala = self.g.registrar_servicio(Sala("S1", "Principal", 50000, 10))

    def reserva(self, codigo="R1"):
        return self.g.crear_reserva(codigo, "12345", "S1", 2)

    def test_entidad_abstracta(self):
        with self.assertRaises(TypeError):
            Entidad("X")

    def test_servicio_abstracto(self):
        with self.assertRaises(TypeError):
            Servicio("S", "Servicio", 10)

    def test_nombre_invalido(self):
        with self.assertRaises(DatoInvalido):
            self.cliente.nombre = ""

    def test_correo_invalido(self):
        with self.assertRaises(DatoInvalido):
            self.cliente.correo = "ana"
        self.assertEqual(self.cliente.correo, "ana@example.com")

    def test_telefono_invalido(self):
        with self.assertRaises(DatoInvalido):
            self.cliente.telefono = "abc"

    def test_documento_invalido(self):
        with self.assertRaises(DatoInvalido):
            Cliente("1", "Ana Pérez", "ana@example.com", "3001234567")

    def test_cliente_duplicado(self):
        with self.assertRaises(Duplicado):
            self.g.registrar_cliente("12345", "Ana Pérez", "ana@example.com", "3001234567")
        self.assertEqual(len(self.g.clientes), 1)

    def test_tarifa_negativa(self):
        with self.assertRaises(DatoInvalido):
            Sala("S2", "Otra sala", -1, 10)

    def test_tarifa_no_finita(self):
        with self.assertRaises(DatoInvalido):
            Sala("S2", "Otra sala", float("inf"), 10)

    def test_capacidad_decimal(self):
        with self.assertRaises(DatoInvalido):
            Sala("S2", "Otra sala", 100, 2.5)

    def test_costo_base(self):
        self.assertEqual(self.sala.calcular_costo(2), 100000)

    def test_costo_impuesto(self):
        self.assertEqual(self.sala.calcular_costo(2, 19), 119000)

    def test_costo_descuento(self):
        self.assertEqual(self.sala.calcular_costo(2, 19, 10), 107100)

    def test_impuesto_invalido(self):
        with self.assertRaises(DatoInvalido):
            self.sala.calcular_costo(2, -1)

    def test_alquiler_dias_enteros(self):
        equipo = Equipo("E1", "Portátil", 80000, "Informática")
        self.assertEqual(equipo.calcular_costo(3), 240000)
        with self.assertRaises(DatoInvalido):
            equipo.calcular_costo(1.5)

    def test_asesoria(self):
        asesoria = Asesoria("A1", "Consultoría", 100000, "Software")
        self.assertEqual(asesoria.calcular_costo(2), 200000)

    def test_asesoria_corta(self):
        with self.assertRaises(DatoInvalido):
            Asesoria("A1", "Consultoría", 100, "Software").calcular_costo(0.5)

    def test_servicio_duplicado(self):
        with self.assertRaises(Duplicado):
            self.g.registrar_servicio(self.sala)

    def test_cliente_inexistente(self):
        with self.assertRaises(NoEncontrado):
            self.g.crear_reserva("R1", "99999", "S1", 1)

    def test_duracion_invalida(self):
        with self.assertRaises(DatoInvalido):
            self.g.crear_reserva("R1", "12345", "S1", 0)
        self.assertEqual(len(self.g.reservas), 0)

    def test_confirmar(self):
        r = self.reserva()
        self.g.confirmar("R1")
        self.assertEqual(r.estado, "Confirmada")
        self.assertFalse(self.sala.disponible)

    def test_confirmacion_doble(self):
        r = self.reserva()
        r.confirmar()
        with self.assertRaises(EstadoIncorrecto):
            r.confirmar()

    def test_servicio_ocupado(self):
        self.reserva().confirmar()
        with self.assertRaises(NoDisponible):
            self.reserva("R2")

    def test_cancelar_pendiente(self):
        r = self.reserva()
        r.cancelar()
        self.assertEqual(r.estado, "Cancelada")
        self.assertTrue(self.sala.disponible)

    def test_cancelar_confirmada(self):
        r = self.reserva()
        r.confirmar()
        r.cancelar()
        self.assertTrue(self.sala.disponible)

    def test_procesar_pendiente(self):
        with self.assertRaises(EstadoIncorrecto):
            self.reserva().procesar()

    def test_procesar_confirmada(self):
        r = self.reserva()
        r.confirmar()
        self.assertEqual(r.procesar(19, 10), 107100)
        self.assertEqual(r.estado, "Procesada")
        self.assertTrue(self.sala.disponible)

    def test_error_encadenado(self):
        r = self.reserva()
        r.confirmar()
        with self.assertRaises(DatoInvalido) as error:
            r.procesar(descuento=101)
        self.assertIsInstance(error.exception.__cause__, DatoInvalido)
        self.assertEqual(r.estado, "Confirmada")

    def test_reserva_duplicada(self):
        self.reserva()
        with self.assertRaises(Duplicado):
            self.reserva()

    def test_simulaciones_y_log(self):
        with tempfile.TemporaryDirectory() as directorio:
            ruta = Path(directorio) / "prueba.log"
            logger = configurar_log(ruta)
            try:
                correctos, total = simular(logger)
                self.assertEqual(correctos, total)
                self.assertGreaterEqual(total, 10)
                contenido = ruta.read_text(encoding="utf-8")
                self.assertIn("ERROR", contenido)
                self.assertIn("Correcto", contenido)
                self.assertIn("Fin de operación", contenido)
            finally:
                for handler in logger.handlers[:]:
                    handler.close()
                    logger.removeHandler(handler)


if __name__ == "__main__":
    unittest.main()


## Simulaciones

In [ ]:
!python simulaciones.py

## Pruebas unitarias

In [ ]:
!python -m unittest -v

## Registro de eventos y errores

In [ ]:
!cat logs/software_fj.log

## Menú (opcional)
Ejecutar `!python main.py` en una celda adicional y seleccionar 0 para salir.